In [ ]:
!pip install -q -U diffusers transformers accelerate peft "torchao>=0.16.0"

from diffusers import StableDiffusionPipeline
import torch
import matplotlib.pyplot as plt
import os

# Cria pastas para organizar as entregas
os.makedirs("avaliacao/parte1_grade_v2", exist_ok=True)
os.makedirs("avaliacao/parte3_memorizacao_v2", exist_ok=True)

pipeline = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", 
    torch_dtype=torch.float16
).to("cuda")

# ==========================================
# DADOS PARA A PARTE 1 & 2 (6 Prompts Fixos)
# ==========================================
prompts_grade = [
    "estilo_rupestre, uma pintura de um animal com chifres em uma parede de rocha",
    "estilo_rupestre, uma pintura de um grupo de pessoas caçando com lanças",
    "estilo_rupestre, o desenho de um pássaro",
    "estilo_rupestre, uma pintura de um animal com chifres e um pássaro",
    "estilo_rupestre, uma pintura de pessoas dançando",
    "estilo_rupestre, uma pintura de uma grande família de animais selvagens"
]

semente = 42
imagens_base = []
imagens_lora = []

print("🎨 [Parte 1] Gerando imagens com o Modelo BASE...")
for texto in prompts_grade:
    img = pipeline(texto, num_inference_steps=30, generator=torch.Generator("cuda").manual_seed(semente)).images[0]
    imagens_base.append(img)

print("🎨 [Parte 1] Gerando imagens com o Modelo LORA...")
# ATENÇÃO: Substitua pelo nome do seu modelo campeão (V1 ou V2)
pipeline.load_lora_weights("LuisCurado/lora-estilo-rupestre")
for texto in prompts_grade:
    img = pipeline(texto, num_inference_steps=30, generator=torch.Generator("cuda").manual_seed(semente)).images[0]
    imagens_lora.append(img)

# Montando a Grade 6x2 exigida
fig, axs = plt.subplots(6, 2, figsize=(10, 24))
for i in range(6):
    axs[i, 0].imshow(imagens_base[i])
    axs[i, 0].set_title(f"Base - Prompt {i+1}")
    axs[i, 0].axis('off')
    
    axs[i, 1].imshow(imagens_lora[i])
    axs[i, 1].set_title(f"LoRA - Prompt {i+1}")
    axs[i, 1].axis('off')

plt.tight_layout()
plt.savefig("avaliacao/parte1_grade_v2/grade_6x2.png")
print("✅ Grade 6x2 salva na pasta 'avaliacao/parte1_grade_v2'!")

# ==========================================
# DADOS PARA A PARTE 3 (10 Prompts do Dataset)
# ==========================================
print("\n🎨 [Parte 3] Gerando 10 imagens para teste de Memorização...")
prompts_treino = [
    "estilo_rupestre, uma rocha com uma pintura rupestre de dois animais com chifres.",
    "estilo_rupestre, uma pintura de um veado com chifres bem grandes em uma rocha",
    "estilo_rupestre, uma rocha com uma pintura onde se e apresentado muitos animais nela, são 4 cabritos e um pastor.",
    "estilo_rupestre, uma rocha com várias pinturas no estilo rupestre, nela tem vários animais e várias representações de pessoas.",
    "estilo_rupestre, uma rocha com uma pintura rupestre na parede, onde é visto os desenhos de um cavalo ao centro, dois touros um de cada lado e no centro inferior vários veados.",
    "estilo_rupestre, uma parede de rocha com a pintura de um pessoa nela, aparentemente atirando um bumerangue e várias pessoas menores atirando flecha em uma pessoa(ser) maior.",
    "estilo_rupestre, uma rocha com uma pintura nela, onde há 10 pessoas nela, na qual 4 estão de ponta cabeça.",
    "estilo_rupestre, uma pintura rupestre de um pássaro em uma rocha.",
    "estilo_rupestre, uma pintura rupestre em uma rocha, onde há 2 pessoas segurando uma rede de caça enquanto outras 4 pessoas encurrala um animal para ser capturado pela rede, estão caçando.",
    "estilo_rupestre, pintura em uma rocha onde ha vária pessoas caçando uma animal bem grande que está ao centro, alguma pessoas estão municiando armas e no canto inferior á um animal menor."
]

for i, texto in enumerate(prompts_treino):
    img = pipeline(texto, num_inference_steps=30, generator=torch.Generator("cuda").manual_seed(semente)).images[0]
    img.save(f"avaliacao/parte3_memorizacao_v2/teste_overfitting_{i+1}.png")
    
print("✅ As 10 imagens para verificação de cópia foram salvas na pasta 'avaliacao/parte3_memorizacao'!")

# Salvando as imagens individuais da Parte 1 no disco para o cálculo do CLIP
for i, (img_b, img_l) in enumerate(zip(imagens_base, imagens_lora)):
    img_b.save(f"avaliacao/parte1_grade_v2/base_{i}.png")
    img_l.save(f"avaliacao/parte1_grade_v2/lora_{i}.png")

In [ ]:
!pip install -q torchmetrics torchvision transformers==4.38.2 matplotlib numpy

import torch
import torchvision.transforms as T
from torchmetrics.multimodal.clip_score import CLIPScore
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

print("Baixando o modelo avaliador CLIP...\n")
# Instancia a métrica exigida no roteiro
metrica = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to("cuda")

prompts = [
    "estilo_rupestre, uma pintura de um animal com chifres em uma parede de rocha",
    "estilo_rupestre, uma pintura de um grupo de pessoas caçando com lanças",
    "estilo_rupestre, o desenho de um pássaro",
    "estilo_rupestre, uma pintura de um animal com chifres e um pássaro",
    "estilo_rupestre, uma pintura de pessoas dançando",
    "estilo_rupestre, uma pintura de uma grande família de animais selvagens"
]

def calcular_scores(pasta, prefixo):
    transformar_em_tensor = T.PILToTensor()
    scores_individuais = []
    
    for i, texto in enumerate(prompts):
        # Agora o caminho da imagem usa a pasta e o prefixo dinamicamente
        caminho_imagem = f"{pasta}/{prefixo}_{i}.png"
        img = Image.open(caminho_imagem).convert("RGB")
        
        tensor_da_imagem = transformar_em_tensor(img).to("cuda")
        score = metrica(tensor_da_imagem, texto)
        scores_individuais.append(score.item())
        
    media = sum(scores_individuais) / len(prompts)
    return scores_individuais, media

print("Calculando as pontuações matemáticas...\n")

# ==========================================
# 1. DEFINIÇÃO DAS PASTAS E PREFIXOS
# ==========================================
# Ajuste os caminhos abaixo conforme a localização exata das suas imagens
pasta_lora_1 = "/content/avaliacao/parte1_grade_v2"
pasta_lora_2 = "/content/avaliacao/parte1_grade" # Caminho da sua nova LoRA

# Lendo as imagens das respectivas pastas
scores_base, media_base = calcular_scores(pasta_lora_1, "base")
scores_lora1, media_lora1 = calcular_scores(pasta_lora_1, "lora")
scores_lora2, media_lora2 = calcular_scores(pasta_lora_2, "lora") # Ajuste o prefixo ("lora2") se salvou com outro nome

print(f"📊 Média BASE:   {media_base:.2f}")
print(f"📊 Média LoRA V1: {media_lora1:.2f}")
print(f"📊 Média LoRA V2: {media_lora2:.2f}")

# ==========================================
# 2. GERAÇÃO DO GRÁFICO 
# ==========================================
print("\nGerando gráfico...")

# Rótulos do eixo X (Prompt 1 a 6 + Média)
labels = [f"Prompt {i+1}" for i in range(len(prompts))] + ["MÉDIA FINAL"]

# Agrupando os dados (notas individuais + nota média no final)
dados_base = scores_base + [media_base]
dados_lora1 = scores_lora1 + [media_lora1]
dados_lora2 = scores_lora2 + [media_lora2]

x = np.arange(len(labels))
width = 0.25  # Largura das barras

fig, ax = plt.subplots(figsize=(14, 7))

# Criando as barras para cada modelo
barras_base = ax.bar(x - width, dados_base, width, label='Modelo Base', color='#7f8c8d')
barras_lora1 = ax.bar(x, dados_lora1, width, label='LoRA 1', color='#2980b9')
barras_lora2 = ax.bar(x + width, dados_lora2, width, label='LoRA 2', color='#e67e22')

# Configurações visuais do gráfico
ax.set_ylabel('CLIPScore', fontsize=12)
ax.set_title('Desempenho no CLIPScore por Prompt e Média Geral', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Função para exibir o número exato em cima de cada barra
def autolabel(barras):
    for bar in barras:
        altura = bar.get_height()
        ax.annotate(f'{altura:.1f}',
                    xy=(bar.get_x() + bar.get_width() / 2, altura),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(barras_base)
autolabel(barras_lora1)
autolabel(barras_lora2)

plt.tight_layout()
plt.show()